# extraction.py — Interactive Test

Runs `extract_pdf` against the sample PDFs in the project root and inspects the output.

Make sure you have installed the requirements first:
```bash
pip install -r backend/requirements.txt
```

In [28]:
import sys
from pathlib import Path

ROOT = Path(".").resolve().parent
PDF_DIR = ROOT / "pdf_documents"
sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("PDF directory:", PDF_DIR)

Project root: C:\Users\Leal\learning\Tractian_case
PDF directory: C:\Users\Leal\learning\Tractian_case\pdf_documents


In [4]:
from core.extraction import extract_pdf, ExtractionResult, PageContent

print("Import OK")

Import OK


## Helper

In [5]:
def load_and_extract(pdf_path: Path) -> ExtractionResult:
    """Read a PDF from disk and run extract_pdf."""
    file_bytes = pdf_path.read_bytes()
    result = extract_pdf(file_bytes, pdf_path.name)
    return result


def print_summary(result: ExtractionResult) -> None:
    print(f"File            : {result.filename}")
    print(f"Total pages     : {result.total_pages}")
    print(f"Extractable text: {result.has_extractable_text}")
    if result.warning:
        print(f"Warning         : {result.warning}")
    print(f"Pages returned  : {len(result.pages)}")
    total_chars = sum(len(p.text) for p in result.pages)
    print(f"Total chars     : {total_chars:,}")

## 1 — List available sample PDFs

In [18]:
pdfs = sorted(PDF_DIR.glob("*.pdf"))
for p in pdfs:
    print(p.name)
pdfs

LB5001.pdf
MN414_0224.pdf
WEG-CESTARI-manual-iom-guia-consulta-rapida-50111652-pt-en-es-web.pdf
WEG-motores-eletricos-guia-de-especificacao-50032749-brochure-portuguese-web.pdf


[WindowsPath('C:/Users/Leal/learning/Tractian_case/pdf_documents/LB5001.pdf'),
 WindowsPath('C:/Users/Leal/learning/Tractian_case/pdf_documents/MN414_0224.pdf'),
 WindowsPath('C:/Users/Leal/learning/Tractian_case/pdf_documents/WEG-CESTARI-manual-iom-guia-consulta-rapida-50111652-pt-en-es-web.pdf'),
 WindowsPath('C:/Users/Leal/learning/Tractian_case/pdf_documents/WEG-motores-eletricos-guia-de-especificacao-50032749-brochure-portuguese-web.pdf')]

## 2 — Extract first PDF and inspect summary

In [23]:
result = load_and_extract(pdfs[3])
print_summary(result)

File            : WEG-motores-eletricos-guia-de-especificacao-50032749-brochure-portuguese-web.pdf
Total pages     : 68
Extractable text: True
Pages returned  : 68
Total chars     : 229,169


## 3 — Inspect individual pages

In [24]:
# Print first 3 pages
for page in result.pages[:3]:
    print(f"\n{'='*60}")
    print(f"Page {page.page_number}  ({len(page.text)} chars)")
    print('='*60)
    print(page.text[:800])


Page 1  (39 chars)
GUIA DE ESPECIFICAÇÃO
MOTORES ELÉTRICOS

Page 2  (1 chars)
2

Page 3  (870 chars)
3
Onde quer que haja progresso, a presença do 
motor elétrico é imprescindível. Desempenhando 
um importante papel para a sociedade, os 
motores são o coração das máquinas modernas, 
por essa razão é necessário conhecer seus 
princípios fundamentais de funcionamento, desde 
a construção até as aplicações. 
O Guia de Especificação de Motores Elétricos 
WEG auxilia de maneira simples e objetiva 
aqueles que compram, vendem e trabalham 
com esses equipamentos, trazendo instruções 
de manuseio, uso e funcionamento dos mais 
diversos tipos de motores.
Na era das máquinas modernas os motores 
elétricos são o combustível da inovação. 
Esse material tem como objetivo apresentar 
à todos os apaixonados pela eletricidade, o 
crescimento contínuo das novas tecnologias, sem 
perder a simplicidade do f


## 4 — Run against all sample PDFs

In [25]:
for pdf in pdfs:
    r = load_and_extract(pdf)
    print_summary(r)
    print("-" * 50)

File            : LB5001.pdf
Total pages     : 2
Extractable text: True
Pages returned  : 2
Total chars     : 6,276
--------------------------------------------------
File            : MN414_0224.pdf
Total pages     : 16
Extractable text: True
Pages returned  : 15
Total chars     : 29,121
--------------------------------------------------
File            : WEG-CESTARI-manual-iom-guia-consulta-rapida-50111652-pt-en-es-web.pdf
Total pages     : 84
Extractable text: True
Pages returned  : 81
Total chars     : 129,541
--------------------------------------------------
File            : WEG-motores-eletricos-guia-de-especificacao-50032749-brochure-portuguese-web.pdf
Total pages     : 68
Extractable text: True
Pages returned  : 68
Total chars     : 229,169
--------------------------------------------------


## 5 — Scanned PDF simulation (empty bytes → warning path)

In [26]:
import fitz

# Build a valid PDF with an image-only page (no text layer) in memory
doc = fitz.open()
doc.new_page()
fake_bytes = doc.tobytes()
doc.close()

scanned_result = extract_pdf(fake_bytes, "fake_scanned.pdf")
print_summary(scanned_result)

action=extract | file=fake_scanned.pdf | no_extractable_text=true


File            : fake_scanned.pdf
Total pages     : 1
Extractable text: False
Pages returned  : 0
Total chars     : 0
